# EXP 3 — full pool for TabPFN via disjoint bagging

우선순위 3. TabPFN은 한 컨텍스트에 100만 행 넘게 못 넣으므로,
**풀을 estimator들에 나눠서** 전체가 참여하게 합니다.

`SUBSAMPLE_SAMPLES=k`를 주면 estimator마다 **서로 겹치지 않는** stratified k행을
가져가고(공유 풀에서 순차로 빼감, `ensemble.py:735-769`),
`n_estimators = ⌈pool/k⌉`이면 풀을 정확히 한 번씩 덮습니다.

**쓰기 전에 반드시 읽을 것**
1. 이건 "1,207만 행으로 학습"이 **아닙니다**. 각 예측기는 여전히 k행에만 조건화되고
   앙상블은 확률 평균(bagging)입니다. → "disjoint bagging으로 전체 풀 100% 커버"
2. **어떤 논문에도 없는 방식**입니다. 위 동작들은 라이브러리에 실재하지만,
   full coverage로 조합한 건 TabPFN 논문에도 참조 IDS 논문에도 README에도 없습니다.
3. stratified는 **비례 배분**이라 tail을 보호하지 않습니다 —
   `web_attacks`는 k=90만에서 약 114행입니다(0813.md의 starvation 재생산).
4. 기존 est=4 run 대비 **노브가 두 개** 움직입니다(앙상블 크기 + 커버리지).
   원인 분리에는 마지막의 **control run**이 필요합니다.

예상 소요: **bot_iot ~1.2h · cic2018 ~2.6h · ton_iot ~3.9h**

배경 문서: `docs/tabpfn_v2_memory_rootcause.html`


## 1. Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 경로
Drive 구조 (이미 만들어 두신 그대로):
```
MyDrive/imbal_cic_tabpfn/
  src/            exp_utils.py, *.ckpt  (+ 아래 .py 2개를 여기 업로드)
  data/           *.pkl
  results/        <- --out-root
  saved_models/   <- --models-dir
```
**src/ 에 올릴 .py 2개**: `nfv3_v3_common.py`, `nfv3_v3_exp3_full_tabpfn_bagging.py`

`exp_utils.py`는 데이터 로더와 클래스별 chronological split 코드라 반드시 필요합니다.


In [ ]:
import os, glob, shutil, sys

DRIVE_ROOT = '/content/drive/MyDrive/imbal_cic_tabpfn'
CODE_DIR   = DRIVE_ROOT + '/src'
DATA_DIR   = DRIVE_ROOT + '/data'
OUT_ROOT   = DRIVE_ROOT + '/results'

# 로컬 작업 트리. 폴더명을 'tabpfn'으로 두면 패키지명과 헷갈리므로 'exp'.
WORK    = '/content/work'
SCRATCH = DRIVE_ROOT + '/tabpfn_cache'   # 세션이 끊겨도 fit 재사용

MODELS_DIR = DRIVE_ROOT + '/saved_models'   # 기본 모드 저장본 ~2-3 GB — Drive OK(업로드 느림)

for d in (SCRATCH, OUT_ROOT, MODELS_DIR, WORK + '/exp', WORK + '/scripts'):
    os.makedirs(d, exist_ok=True)

for label, path in [('CODE_DIR', CODE_DIR), ('DATA_DIR', DATA_DIR)]:
    ok = os.path.isdir(path)
    print(f'{label:10s} {path}   exists={ok}')
    if not ok:
        raise FileNotFoundError(path + ' 가 없습니다. DRIVE_ROOT를 확인하세요.')
print(f'{"MODELS_DIR":10s} {MODELS_DIR}')


## 3. 패키지 설치 — **실패하면 여기서 멈춥니다**
이전 버전은 `!pip -q install ... | tail -1`이라 실패가 조용히 넘어갔고,
그 결과 실행 단계에서 `ModuleNotFoundError: No module named 'tabpfn'`이 났습니다.
여기서는 import까지 확인하고, 버전을 로컬과 맞추려 `tabpfn==8.2.0`으로 고정합니다.


In [ ]:
import importlib, subprocess, sys

def ensure(module, spec):
    try:
        importlib.import_module(module)
        print(f'  {module}: 이미 설치됨')
        return
    except ImportError:
        pass
    print(f'  installing {spec} ...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-4000:]); print(r.stderr[-4000:])
        raise RuntimeError(f'pip install {spec} 실패 — 위 로그를 보세요')
    importlib.invalidate_caches()
    importlib.import_module(module)
    print(f'  {module}: 설치 완료')

ensure('tabpfn', 'tabpfn==8.2.0')
ensure('xgboost', 'xgboost')

import tabpfn, xgboost
print()
print('python  :', sys.executable)
print('tabpfn  :', tabpfn.__version__, '\n          ', tabpfn.__file__)
print('xgboost :', xgboost.__version__)


## 4. GPU 확인 — **T4면 여기서 멈춥니다**


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경에서 GPU를 고르세요.')

name  = torch.cuda.get_device_name(0)
cap   = torch.cuda.get_device_properties(0).total_memory / 1024**3
major = torch.cuda.get_device_capability(0)[0]
print(f'GPU: {name}   {cap:.1f} GiB   sm{major}x')

if major < 8:
    raise RuntimeError(
        f'{name}(sm{major}x)에서는 FlashAttention이 안 돕니다(sm80+ 필요). '
        'MATH 백엔드로 떨어져 메모리가 컨텍스트 길이의 제곱이 되고, 이 노트북의 '
        '메모리 예측이 전부 무효가 됩니다. A100 또는 L4로 런타임을 다시 잡으세요.')
print('OK — 선형 메모리 모델이 성립하는 하드웨어입니다.')

# 조각화 방지: 없으면 들어갈 run도 OOM 납니다 (실측 2.7 GiB 손실)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('PYTORCH_CUDA_ALLOC_CONF =', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


## 5. 코드 배치
v3 스크립트는 `REPO_ROOT/scripts/exp_utils.py`를 import합니다
(`REPO_ROOT` = 스크립트 부모의 부모). 그 구조를 로컬에 만듭니다.
Drive의 `src/` 아래 어느 하위 폴더에 있든 찾아냅니다.


In [ ]:
SCRIPT = 'nfv3_v3_exp3_full_tabpfn_bagging.py'
NEEDED = ['nfv3_v3_common.py', SCRIPT, 'exp_utils.py']

missing = []
for name in NEEDED:
    hits = glob.glob(os.path.join(CODE_DIR, '**', name), recursive=True)
    if not hits:
        missing.append(name); continue
    dst = (WORK + '/scripts/' if name == 'exp_utils.py' else WORK + '/exp/') + name
    shutil.copy(hits[0], dst)
    print(f'  {name:32s} <- {hits[0]}')

if missing:
    raise FileNotFoundError(
        f'{CODE_DIR} 아래에서 못 찾은 파일: {missing}\n'
        '로컬 repo의 tabpfn/nfv3_v3_*.py 와 scripts/exp_utils.py 를 업로드하세요.')
print('\nOK')


## 6. 체크포인트
`src/` 안의 `.ckpt`를 찾습니다. 없으면 HF에서 받아 Drive에 캐시합니다.


In [ ]:
CKPT_NAME = 'tabpfn-v3-classifier-v3_20260417_multiclass.ckpt'
hits = glob.glob(os.path.join(CODE_DIR, '**', CKPT_NAME), recursive=True)
if hits:
    CKPT = hits[0]
else:
    from huggingface_hub import hf_hub_download
    src = hf_hub_download(repo_id='Prior-Labs/tabpfn_3', filename=CKPT_NAME)
    CKPT = os.path.join(CODE_DIR, CKPT_NAME)
    shutil.copy(src, CKPT)
print(CKPT, f'{os.path.getsize(CKPT)/1e6:.0f} MB')


## 7. 실행 설정


In [ ]:
TARGET = 'cic2018'   # cic2018 | bot_iot | ton_iot | *_capped | cic2017_full

# exp3은 기본 모드(캐시 X — 캐시가 k×estimator라 52~70 GB가 되어버림).
# 컨텍스트 90만 기준 base 15.74 GiB, 20.5 GiB 안전선까지 여유 = test 약 414,000행
#   bot_iot_capped  70,722 -> 70722   (1배치)   cic2018_capped 120,518 -> 120518 (1배치)
#   ton_iot_capped 161,999 -> 161999  (1배치)   cic2017_full   211,322 -> 211322 (1배치)
#   bot_iot        310,722 -> 310722  (1배치)
#   cic2018        440,276 -> 220138  (2배치)
#   ton_iot        659,725 -> 329863  (2배치)
K          = 900_000   # estimator당 컨텍스트
TEST_BATCH = 220138    # cic2018 기준. 타겟에 맞게 위 표에서 고르세요


## 8. 실행
`sys.executable`로 돌립니다 — Colab에서 `python`은 pip가 설치한 인터프리터와
다를 수 있고, 그게 `ModuleNotFoundError: No module named 'tabpfn'`의 원인이었습니다.


In [ ]:
cmd = [
    sys.executable, SCRIPT,
    '--target-dataset', TARGET,
    '--subsample-samples', str(K),
    '--test-batch-size', str(TEST_BATCH),
    # --n-estimators 기본 0 = auto(ceil(pool/k)); --ignore-pretraining-limits 는 강제됨
    '--data-dir', DATA_DIR,
    '--out-root', OUT_ROOT,
    '--model-path', CKPT,
    '--resume-dir', SCRATCH,
    '--models-dir', MODELS_DIR,
]
print(' '.join(cmd))


In [ ]:
import subprocess, sys

def run(command):
    """Stream the run, capture its output dir, FAIL LOUDLY on a bad exit.

    An earlier version of this notebook globbed for the newest results dir
    afterwards, so a crashed run silently displayed a PREVIOUS run's numbers.
    """
    p = subprocess.Popen(command, cwd=WORK + '/exp', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ})
    captured = []
    for line in p.stdout:
        sys.stdout.write(line)
        captured.append(line)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(
            f'실행 실패 (exit code {p.returncode}) — 위 로그를 보세요. '
            '아래 결과 셀은 실행하지 마세요(예전 run을 읽게 됩니다).')
    out = [l.split('Wrote ', 1)[1].strip() for l in captured if l.startswith('Wrote ')]
    if not out:
        raise RuntimeError("'Wrote <dir>' 줄이 없습니다 — 아티팩트가 안 만들어졌습니다.")
    return out[-1]

RUN_DIR = run(cmd)
print('\nrun dir:', RUN_DIR)


## 9. 결과
위 셀이 만든 **바로 그** 디렉터리(`RUN_DIR`)만 읽습니다.


In [ ]:
import pandas as pd, json

t = pd.read_csv(os.path.join(RUN_DIR, 'per_class_metrics.csv'))
avg = ['macro_avg', 'weighted_avg', 'tail_avg']
display(t[t['class'].isin(avg)].pivot(index='class', columns='method', values='f1'))
print()
display(t[~t['class'].isin(avg)].pivot(index='class', columns='method',
                                       values=['f1', 'support']))


In [ ]:
print(json.dumps(json.load(open(os.path.join(RUN_DIR, 'timings.json')))[0], indent=2))


`per_class_metrics.csv` · `split_audit.csv` · `train_test_drift_diagnostic.csv` ·
`args.json` · `timings.json` 가 `RUN_DIR`에 저장됩니다.

⚠️ `train_test_drift_diagnostic.csv`의 `median_abs_z`는 `small_train_warning=True`인
행에서 신뢰할 수 없습니다(표준편차를 수십 행에서 추정). `manuscript/report/0817.md` 참조.


## 10. Control run — 원인 분리용
위 run은 기존 est=4 대비 **앙상블 크기와 커버리지가 동시에** 바뀝니다.
차이를 커버리지 탓으로 돌리려면 같은 k행을 **모든** estimator에 넣은 대조군이 필요합니다.
`treatment − control = 커버리지 효과` (CLAUDE.md M6).


In [ ]:
N_EST = json.load(open(os.path.join(RUN_DIR, 'timings.json')))[0]['n_estimators']
print('control run with n_estimators =', N_EST)

ctrl = [sys.executable, SCRIPT,
        '--target-dataset', TARGET,
        '--max-train-samples', str(K),   # 모든 estimator가 같은 k행을 봄
        '--subsample-samples', '0',      # 커버리지 없음
        '--n-estimators', str(N_EST),
        '--test-batch-size', str(TEST_BATCH),
        '--data-dir', DATA_DIR, '--out-root', OUT_ROOT, '--model-path', CKPT,
        '--resume-dir', SCRATCH, '--models-dir', MODELS_DIR]
CTRL_DIR = run(ctrl)
print('\ncontrol run dir:', CTRL_DIR)


In [ ]:
a = pd.read_csv(os.path.join(CTRL_DIR, 'per_class_metrics.csv'))
b = pd.read_csv(os.path.join(RUN_DIR, 'per_class_metrics.csv'))
m = (a[a['method'] == 'tabpfn_v3'].set_index('class')['f1'].rename('control').to_frame()
     .join(b[b['method'] == 'tabpfn_v3'].set_index('class')['f1'].rename('full_coverage')))
m['delta'] = m['full_coverage'] - m['control']
display(m.sort_values('delta', ascending=False))
